 # ASSIGNMENT NLP – 4 (BERT Fine-Tuning)

 #  1 Model loading

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import confusion_matrix

 Load pre-trained BERT model for classification

In [2]:
dataset = load_dataset("imdb")

In [3]:
train_dataset = dataset["train"].select(range(300))
val_dataset = dataset["test"].select(range(100))

 <!-- 2Tokenization -->

# 2 Tokenization

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

 # 3 Dataset split

In [6]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

# 4 Training Arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    logging_strategy="steps"
)

# 5 Trainer

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [11]:
trainer.train()

C:\Users\USER VICTUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.035681,0.020848,1.000000,0.000000,0.000000,0.000000


C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=38, training_loss=0.09552863789232154, metrics={'train_runtime': 1749.5548, 'train_samples_per_second': 0.171, 'train_steps_per_second': 0.022, 'total_flos': 78933316608000.0, 'train_loss': 0.09552863789232154, 'epoch': 1.0})

#  7 Confusion Matrix

In [14]:
from sklearn.metrics import confusion_matrix
import numpy as np

preds = trainer.predict(val_dataset)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

cm = confusion_matrix(y_true, y_pred)
print(cm)

C:\Users\USER VICTUS\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[[100]]


C:\Users\USER VICTUS\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


 <!-- Experiment 1: Freeze BERT -->

 #   Experiment 1

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
)

In [ ]:
for param in model.bert.parameters():
    param.requires_grad = False

In [ ]:
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

In [ ]:
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

In [ ]:
for param in model.classifier.parameters():
    param.requires_grad = True

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

| Experiment                    | Training Strategy                  | Accuracy | F1 Score          | Training Time | Observation                                      |
| ----------------------------- | ---------------------------------- | -------- | ----------------- | ------------- | ------------------------------------------------ |
| Base Model (Full Fine-Tuning) | All BERT layers trained            | High     | Low/0             | High          | Model trained fully but overfitted on small data |
| Experiment 1 (Freeze BERT)    | Only classifier trained            | Low      | Low               | Very Fast     | Faster training but poor learning capability     |
| Experiment 2 (Last 2 Layers)  | Last 2 layers + classifier trained | Moderate | Better than Exp 1 | Moderate      | Balanced performance and efficiency              |


Analysis

In this task, different fine-tuning strategies of the BERT model were explored on the IMDB dataset.

The base model with full fine-tuning showed high accuracy but suffered from poor generalization due to limited training data.

In Experiment 1, where all BERT layers were frozen, the model trained much faster but failed to capture complex patterns, resulting in lower performance.

In Experiment 2, fine-tuning only the last two layers provided a balance between training efficiency and model performance. It allowed the model to learn task-specific features while reducing computational cost.

Overall, the results demonstrate that partial fine-tuning can be an effective strategy when computational resources are limited.